# **Recommendation**

This notebook focuses on building a Recommendation System to suggest relevant tourist attractions to users based on historical interaction data. Understand how recommendation systems work in a tourism context. Build a recommendation engine using user ratings and attraction data. Generate Top-N attraction recommendations. Prepare the model logic for Streamlit deployment

# **Data Preparation for Collaborative Filtering**

Collaborative Filtering relies on user–item interactions rather than explicit item features.
To build such a model, we first need a dataset that captures which user interacted with which attraction and how they rated it.

**In this step:**

The Transaction dataset is loaded to access user visit history and ratings.

Only the essential columns — UserId, AttractionId, and Rating — are selected.

This reduced dataset forms the foundation for constructing a User–Attraction interaction matrix, which is the core input for collaborative filtering algorithms.

**This step is necessary because:**

Collaborative filtering models do not require attraction metadata

They learn user preferences purely from past behavior and ratings

Removing unnecessary columns improves computational efficiency and model clarity

In [2]:
import pandas as pd

transaction = pd.read_excel("../dataset/Transaction.xlsx") # We load the transactions dataset
cf_df = transaction[['UserId', 'AttractionId', 'Rating']] # We only select the required columns from the transaction dataset

# **Creating the User–Attraction Interaction Matrix**

To apply Collaborative Filtering, we must represent user behavior in a structured numerical format.
This is done by converting transactional data into a User–Attraction matrix.

**In this step:**

Each row represents a user

Each column represents an attraction

Each cell value represents the average rating given by a user to an attraction

In [3]:
user_item_matrix = cf_df.pivot_table( # creating a matrix having using interactions
    index='UserId', # every row is one user
    columns='AttractionId', # every column is one attraction
    values='Rating', # every cell or value is average rating
    aggfunc='mean'
)

user_item_matrix = user_item_matrix.fillna(0) # filling the empty values (ratings) with 0

print("Users:", user_item_matrix.shape[0]) # depicts the first column (users)
print("Attractions:", user_item_matrix.shape[1]) # depicts the second column (attractions)

Users: 33530
Attractions: 30


# **Loading the Cleaned Dataset**

In this step, we load the final cleaned tourism dataset that was generated during the data preparation and feature engineering phase.

In [4]:
final_df = pd.read_csv("../dataset/cleaned_tourism_data.csv")

final_df.head() # depicting the first 5 rows

,TransactionId,UserId,VisitYear,VisitMonth,AttractionId,Rating,CityName,Country,Continent,Attraction,AttractionType,AttractionAddress,VisitMode,Region,user_avg_rating,user_total_visits,attraction_avg_rating,attraction_popularity,season
0,3,70456,2022,10,640,5,Guildford,Australia,Europe,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Couples,Australia,5.0,1,4.267086,13198,Fall
1,8,7567,2022,10,640,5,Ontario,Canada,America,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Friends,Northern America,5.0,1,4.267086,13198,Fall
2,9,79069,2022,10,640,5,Brazil,United States,America,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Family,Northern America,5.0,1,4.267086,13198,Fall
3,10,31019,2022,10,640,3,Zurich,Canada,Europe,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Family,Northern America,3.0,2,4.267086,13198,Fall
4,15,43611,2022,10,640,3,Manchester,United States,Europe,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,"Jl. Monkey Forest, Ubud 80571 Indonesia",Couples,Northern America,3.0,3,4.267086,13198,Fall


# **Content-Based Recommendation: Attraction Similarity**

This step implements a Content-Based Recommendation System by measuring similarity between attractions based on their intrinsic features such as type, region, country, popularity, and average rating.

Unlike collaborative filtering, content-based systems:

Do not depend on other users

Recommend attractions that are similar in characteristics

Work well even for new users (cold-start problem)

In [5]:
# Creating the table with only the attraction related features
attraction_features = final_df[[
    'AttractionId',
    'Attraction',
    'AttractionType',
    'Region',
    'Country',
    'attraction_avg_rating',
    'attraction_popularity'
]].drop_duplicates(subset='AttractionId') # Storing only the unique attractions 


# Saving the Attraction ID and Name, Which are not used in similarity calculation, only for:
# 1. Naming rows and columns
# 2. Displaying recommendations to users
ids = attraction_features[['AttractionId', 'Attraction']]



# Remove identifiers
# Keep only features that describe an attraction: Type, Location, Rating, Popularity
# This is the data used to compute similarity
features = attraction_features.drop(
    columns=['AttractionId', 'Attraction']
)


# Converting the text columns into numbers
features_encoded = pd.get_dummies(
    features,
    columns=['AttractionType', 'Region', 'Country'],
    drop_first=False
)



# Scaling the numerical features
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

features_encoded[['attraction_avg_rating', 'attraction_popularity']] = scaler.fit_transform(
    features_encoded[['attraction_avg_rating', 'attraction_popularity']]
)


# Cosine similarity only works on numbers
# This line removes any leftover text columns (if any)
feature_matrix = features_encoded.select_dtypes(
    include=['int64', 'float64', 'bool']
)



# Calculating cosine similarity to calculate how similar to two attractions
from sklearn.metrics.pairwise import cosine_similarity

# Import cosine similarity function (1 - Exactly Similar and 0 - Completely Opposite)
similarity_matrix = cosine_similarity(feature_matrix)


# Converts similarity matrix into a readable table
# Rows & columns are attraction names
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=ids['Attraction'],
    columns=ids['Attraction']
)


# Get similar attractions similar to "Sacred Monkey Foret Sanctuary"
similarity_df.loc[
    "Sacred Monkey Forest Sanctuary"
].sort_values(ascending=False).head(10) # Sorting top 10 attractions with thier similarity scores in descending order

Attraction
Sacred Monkey Forest Sanctuary        1.000000
Sanur Beach                           0.663786
Goa Cina Beach                        0.616489
Sempu Island                          0.364305
Waterbom Bali                         0.261330
Tegalalang Rice Terrace               0.208002
Uluwatu Temple                        0.170696
Tanah Lot Temple                      0.167895
Mount Semeru Volcano                  0.158465
Bromo Tengger Semeru National Park    0.155694
Name: Sacred Monkey Forest Sanctuary, dtype: float64

# **Generating Top-N Attraction Recommendations**

After computing similarity scores between attractions, the next step is to convert these scores into meaningful recommendations that can be presented to users.

This function retrieves the Top-N most similar attractions for a given attraction using the precomputed similarity matrix.

In [6]:
def recommend_attractions(attraction_name, top_n=5): # Creating a function to recommend the top 5 attractions
    if attraction_name not in similarity_df.index: # This block will be executed if there are no similar attractions
        return "Attraction not found" # Returns this statement
    
    # Sort the similar attractions by similarity scores in descending order
    similar_scores = similarity_df.loc[attraction_name] \
                                  .sort_values(ascending=False)
    
    # Excluding the attraction itself
    recommendations = similar_scores.iloc[1:top_n+1]
    
    return recommendations # return the recommendations

    
recommend_attractions( # calling the function with the attraction and top_n=5 as the parameters
    "Sacred Monkey Forest Sanctuary",
    top_n=5
)

Attraction
Sanur Beach                0.663786
Goa Cina Beach             0.616489
Sempu Island               0.364305
Waterbom Bali              0.261330
Tegalalang Rice Terrace    0.208002
Name: Sacred Monkey Forest Sanctuary, dtype: float64

# **Saving Models**

In [7]:
similarity_df.to_pickle("../models/similarity_df.pkl") # Saving the dataframe which contains similarity scores
# To recommend similar attractions

print("Sucessfuly Saved the Similarity Data Frame")

Sucessfuly Saved the Similarity Data Frame
